In [180]:
import torch

In [181]:
## read the file : shakes.txt
with open("shakes.txt", "r") as f:
    text = f.read()

In [182]:
print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [183]:
# get the vocab
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [184]:
print(vocab_size)

print("".join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [185]:
#create mapping from char to int
stoi = {strs:i for i,strs in enumerate(chars)}
itos = {i:strs for i,strs in enumerate(chars)}

In [186]:
encode = lambda inp_txt : [stoi[i] for i in inp_txt]
decode = lambda inp_num : "".join([itos[i] for i in inp_num])

In [187]:
decode(encode("hello there"))

'hello there'

In [188]:
##convert the input into data tensor
import torch
inputs = "hello World"

data = torch.tensor(encode(text))

data

tensor([18, 47, 56,  ..., 45,  8,  0])

In [189]:
## split the data
n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]


In [190]:
block_size = 8 
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [191]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"The context is : {context} and target is : {target}")
    

The context is : tensor([18]) and target is : 47
The context is : tensor([18, 47]) and target is : 56
The context is : tensor([18, 47, 56]) and target is : 57
The context is : tensor([18, 47, 56, 57]) and target is : 58
The context is : tensor([18, 47, 56, 57, 58]) and target is : 1
The context is : tensor([18, 47, 56, 57, 58,  1]) and target is : 15
The context is : tensor([18, 47, 56, 57, 58,  1, 15]) and target is : 47
The context is : tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target is : 58


In [192]:
torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([train_data[i:i+block_size] for i in ix])
    y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])
    return x , y


In [193]:
xb, yb = get_batch("train")

xb, yb

(tensor([[57,  1, 46, 47, 57,  1, 50, 53],
         [ 1, 58, 46, 43, 56, 43,  1, 41],
         [17, 26, 15, 17, 10,  0, 32, 53],
         [57, 58,  6,  1, 61, 47, 58, 46]]),
 tensor([[ 1, 46, 47, 57,  1, 50, 53, 60],
         [58, 46, 43, 56, 43,  1, 41, 39],
         [26, 15, 17, 10,  0, 32, 53,  1],
         [58,  6,  1, 61, 47, 58, 46,  0]]))

In [194]:
# for b in range(batch_size):
#     for t in range(block_size):
#         context = xb[b, :t+1]
#         target = yb[b, t]
#         print(f"The context : {context.tolist()} and target : {target}")

In [202]:
## Biagram model
import torch
from torch import nn
from torch.nn import functional as F 

class BaigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)   
        
    def forward(self, idx, target=None):
        B,T = idx.shape
        logits = self.token_embedding_table(idx) ## idx(B,T)(4,8) --> B,T,C (4,8,65)

        if target is None:
            return logits   
        else:     
            B,T,C = logits.shape
            logits = logits.view(B*T, C) # (32,65)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
            return logits, loss
    
    def generate(self, idx, max_new_tokens):
        
        for _ in range(max_new_tokens):
            logits= self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            print("sam", idx_next)

            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx
            
    

m = BaigramModel(vocab_size=vocab_size)
logits , loss= m(xb,yb)
logits.shape , loss 


(torch.Size([32, 65]), tensor(4.8798, grad_fn=<NllLossBackward0>))

In [203]:
inputs =  torch.zeros(size=(1,1), dtype=torch.int64)
print(inputs.shape)

print(decode(m.generate(inputs, max_new_tokens=100)[0].tolist()))

torch.Size([1, 1])
sam tensor([[14]])
sam tensor([[30]])
sam tensor([[44]])
sam tensor([[33]])
sam tensor([[6]])
sam tensor([[20]])
sam tensor([[26]])
sam tensor([[49]])
sam tensor([[8]])
sam tensor([[37]])
sam tensor([[48]])
sam tensor([[18]])
sam tensor([[33]])
sam tensor([[11]])
sam tensor([[9]])
sam tensor([[21]])
sam tensor([[23]])
sam tensor([[37]])
sam tensor([[48]])
sam tensor([[29]])
sam tensor([[64]])
sam tensor([[29]])
sam tensor([[59]])
sam tensor([[8]])
sam tensor([[18]])
sam tensor([[44]])
sam tensor([[11]])
sam tensor([[58]])
sam tensor([[2]])
sam tensor([[6]])
sam tensor([[17]])
sam tensor([[15]])
sam tensor([[42]])
sam tensor([[43]])
sam tensor([[35]])
sam tensor([[56]])
sam tensor([[42]])
sam tensor([[26]])
sam tensor([[24]])
sam tensor([[46]])
sam tensor([[7]])
sam tensor([[26]])
sam tensor([[9]])
sam tensor([[56]])
sam tensor([[8]])
sam tensor([[63]])
sam tensor([[17]])
sam tensor([[34]])
sam tensor([[35]])
sam tensor([[15]])
sam tensor([[57]])
sam tensor([[33]])
sa

In [197]:
## Training Loop
optimizer = torch.optim.AdamW(m.parameters(), lr=0.001)
for i in range(1000):
    xb,yb= get_batch("train")
    _, loss = m(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(loss.item())
    
    

4.63068962097168
4.669943332672119
4.630727767944336
4.626162528991699
4.870527744293213
4.658463001251221
4.86734676361084
4.914863109588623
4.646171569824219
4.670914173126221
4.823601245880127
4.704941749572754
4.743868827819824
4.741481781005859
4.6431732177734375
4.578514099121094
4.773801326751709
4.827340602874756
4.4934821128845215
4.900815486907959
4.769870281219482
4.63333797454834
4.463449478149414
4.80849027633667
4.669804096221924
4.644983291625977
4.825366497039795
4.77868127822876
4.680816173553467
4.792464733123779
4.935272693634033
4.517582893371582
4.411751747131348
4.713565826416016
4.546159267425537
4.592123985290527
4.5395402908325195
4.692530155181885
4.427865982055664
4.461569309234619
4.8216776847839355
4.666953086853027
4.332099437713623
4.581460952758789
4.6208367347717285
4.358381748199463
4.924615859985352
4.397426128387451
4.563884258270264
4.56939172744751
4.919597625732422
4.551027774810791
4.442801475524902
4.602378845214844
4.73757791519165
4.6290278434

In [198]:
torch.arange(8)

tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [199]:
a = torch.rand(size=(2,3))
a

tensor([[0.6952, 0.1107, 0.0868],
        [0.3466, 0.6095, 0.3224]])

In [200]:
b = torch.rand(size=(2,3))
b


tensor([[0.6627, 0.5688, 0.8732],
        [0.4518, 0.2281, 0.3317]])

In [201]:
a+b

tensor([[1.3579, 0.6794, 0.9600],
        [0.7984, 0.8376, 0.6541]])